# TrialOutcome M1 -- Dataset Audit

Row-count funnel, null rates, class balance, and the `has_results` leakage check, run directly against `marts.fct_trials` (same filters as `domains/pharma/dataset_builder.py`).

In [1]:
import os
import sys
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

REPO_ROOT = Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT))
load_dotenv(REPO_ROOT / ".env")

from domains.pharma.dataset_builder import _load_config

config = _load_config(REPO_ROOT / "domains" / "pharma" / "config.yaml")
url = (
    f"postgresql+psycopg2://{os.environ['POSTGRES_USER']}:{os.environ['POSTGRES_PASSWORD']}"
    f"@{os.environ['POSTGRES_HOST']}:{os.environ['POSTGRES_PORT']}/{os.environ['POSTGRES_DB']}"
)
engine = create_engine(url)
filters = config["filters"]
filters

{'phase_in': ['PHASE2', 'PHASE3', 'PHASE2|PHASE3'],
 'overall_status_in': ['COMPLETED', 'TERMINATED', 'WITHDRAWN', 'SUSPENDED'],
 'start_date_min': '1990-01-01',
 'start_date_max': 'today'}

## 1. Row-count funnel

Interventional (implied by non-null `phase`) -> Phase 2/3 -> label-eligible status -> plausible `start_date`.

In [2]:
funnel_sql = text(
    """
    SELECT 'all_fct_trials' AS stage, COUNT(*) AS n FROM marts.fct_trials
    UNION ALL
    SELECT 'phase_2_3', COUNT(*) FROM marts.fct_trials WHERE phase = ANY(:phase_in)
    UNION ALL
    SELECT 'phase_2_3_and_label_status', COUNT(*) FROM marts.fct_trials
      WHERE phase = ANY(:phase_in) AND overall_status = ANY(:status_in)
    UNION ALL
    SELECT 'plausible_start_date', COUNT(*) FROM marts.fct_trials
      WHERE phase = ANY(:phase_in) AND overall_status = ANY(:status_in)
        AND start_date >= :start_date_min AND start_date <= CURRENT_DATE
    """
)
params = {
    "phase_in": filters["phase_in"],
    "status_in": filters["overall_status_in"],
    "start_date_min": filters["start_date_min"],
}
with engine.connect() as conn:
    funnel = pd.read_sql(funnel_sql, conn, params=params)
funnel

,stage,n
0,all_fct_trials,594543
1,phase_2_3,114379
2,plausible_start_date,78115
3,phase_2_3_and_label_status,79334


## 2. Null rates on the final filtered set

In [3]:
null_sql = text(
    """
    SELECT
      COUNT(*) AS total,
      COUNT(*) FILTER (WHERE enrollment_count IS NULL) AS null_enrollment_count,
      COUNT(*) FILTER (WHERE num_primary_outcomes IS NULL) AS null_num_primary_outcomes,
      COUNT(*) FILTER (WHERE num_sites IS NULL) AS null_num_sites,
      COUNT(*) FILTER (WHERE allocation IS NULL) AS null_allocation,
      COUNT(*) FILTER (WHERE masking IS NULL) AS null_masking,
      COUNT(*) FILTER (WHERE has_dmc IS NULL) AS null_has_dmc,
      COUNT(*) FILTER (WHERE eligibility_criteria IS NULL) AS null_eligibility_criteria,
      COUNT(*) FILTER (WHERE start_date IS NULL) AS null_start_date,
      COUNT(*) FILTER (WHERE sponsor_key IS NULL) AS null_sponsor_key
    FROM marts.fct_trials
    WHERE phase = ANY(:phase_in) AND overall_status = ANY(:status_in)
      AND start_date >= :start_date_min AND start_date <= CURRENT_DATE
    """
)
with engine.connect() as conn:
    null_counts = pd.read_sql(null_sql, conn, params=params)
total = null_counts.loc[0, "total"]
null_rates = (null_counts.drop(columns="total") / total).T.rename(columns={0: "null_rate"})
null_rates["null_count"] = null_counts.drop(columns="total").T[0]
null_rates.sort_values("null_rate", ascending=False)

,null_rate,null_count
null_has_dmc,0.207707,16225
null_num_sites,0.084260,6582
null_num_primary_outcomes,0.035038,2737
null_masking,0.023184,1811
null_allocation,0.021955,1715
null_enrollment_count,0.015656,1223
null_eligibility_criteria,0.000230,18
null_start_date,0.000000,0
null_sponsor_key,0.000000,0


## 3. Final class balance

Label is derived from `overall_status` (see `config.yaml` -- `fct_trials.is_terminated` is NOT used directly, it undercounts WITHDRAWN/SUSPENDED as negative).

In [4]:
balance_sql = text(
    """
    SELECT
      (overall_status = ANY(:positive_statuses)) AS label,
      COUNT(*) AS n,
      ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct
    FROM marts.fct_trials
    WHERE phase = ANY(:phase_in) AND overall_status = ANY(:status_in)
      AND start_date >= :start_date_min AND start_date <= CURRENT_DATE
    GROUP BY 1
    """
)
balance_params = dict(params, positive_statuses=config["label"]["positive_statuses"])
with engine.connect() as conn:
    balance = pd.read_sql(balance_sql, conn, params=balance_params)
balance

,label,n,pct
0,False,62643,80.19
1,True,15472,19.81


## 4. `has_results` leakage check

If any currently RECRUITING/ACTIVE trial already has `has_results = true`, the column is populated at some point before/independent of trial completion and is safe to use as a design feature. If the count is zero, it would mean `has_results` only ever becomes true after completion -- pure post-outcome leakage -- and the column must be dropped.

In [5]:
leakage_sql = text(
    """
    SELECT COUNT(*) AS n_recruiting_active_with_results
    FROM marts.fct_trials
    WHERE has_results = true AND overall_status IN ('RECRUITING', 'ACTIVE_NOT_RECRUITING')
    """
)
with engine.connect() as conn:
    leakage_check = pd.read_sql(leakage_sql, conn)
leakage_check

,n_recruiting_active_with_results
0,1228


**Conclusion:** the count above is > 0 (1,228 RECRUITING/ACTIVE_NOT_RECRUITING trials as of the 2026-07-30 build) -- `has_results` is populated for a meaningful share of trials before they reach a terminal status, so it is retained as a design feature. It is still worth flagging as a discussion point rather than fully resolved: the base rate of `has_results=true` is presumably far higher among COMPLETED/TERMINATED trials, so the feature likely correlates with trial maturity even where it isn't strictly leaking future information.